# Adaptive Spatial-Temporal Graph Attention Network - Google Colab Training Notebook

This notebook provides the setup and pipeline to train the Spatial-Temporal GAT model with Leave-One-Subject-Out (LOSO) cross-validation on Google Colab, using GPU acceleration.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install PyTorch Geometric and Dependencies

In [ ]:
# Check PyTorch version to install matching PyG binaries
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Install PyG and common packages
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-geometric
!pip install opencv-python mediapipe streamlit plotly pandas scipy scikit-learn fastdtw pyyaml tqdm

## 3. Extract IntelliRehabDS Dataset
Upload the `SkeletonData` folder to your Google Drive, then copy and extract it to the local Colab runtime for fast disk reads.

In [ ]:
# Copy data from Drive and unzip
# Adjust the drive source path to match where you stored the zipped dataset
!mkdir -p /content/dataset
!cp '/content/drive/MyDrive/RehabAI/SkeletonData.zip' /content/dataset/
!unzip -q /content/dataset/SkeletonData.zip -d /content/dataset/
print("Dataset extracted successfully.")

## 4. Modify Configuration for Colab
We can dynamically read the `config.yaml` file and point it to the Colab dataset directory.

In [ ]:
import yaml

config_path = 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Point paths to colab runtime dataset folder
config['paths']['data_dir'] = '/content/dataset/SkeletonData/RawData'
config['training']['num_workers'] = 2
config['training']['epochs'] = 80
config['training']['batch_size'] = 16

with open(config_path, 'w') as f:
    yaml.dump(config, f)
print("Updated configuration for Google Colab environment.")

## 5. Execute Training Loop
Run Leave-One-Subject-Out Cross-Validation. This trains models sequentially for each left-out subject.

In [ ]:
# Run training using the train.py script
!python train.py --epochs 80 --batch-size 16 --device cuda

## 6. Save Checkpoints to Google Drive
Copy final checkpoints, attention outputs, and experiment logging to Google Drive to ensure they are persistent.

In [ ]:
# Copy checkpoints and logs to persistent Google Drive storage
!mkdir -p '/content/drive/MyDrive/RehabAI/checkpoints/'
!mkdir -p '/content/drive/MyDrive/RehabAI/logs/'

!cp -r checkpoints/* '/content/drive/MyDrive/RehabAI/checkpoints/'
!cp -r logs/* '/content/drive/MyDrive/RehabAI/logs/'
print("Training checkpoints and logs backed up to Google Drive.")